In [10]:
from regrid import regrid_fast 
from regrid import regrid
import datetime
import numpy as np
from netCDF4 import Dataset
import xarray as xr
import tqdm
import os

In [2]:
datapath_aux = '/Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/'
ease_lons = np.load(f'{datapath_aux}lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}lat_cent.npy')

### icetype

In [5]:
datapath_in = '/Users/carmennab/Dropbox/OI_PolarSnow/data/NSIDC_icetype/'

years = np.arange(2010,2023)

for year in tqdm.tqdm(years):
    
    d1 = datetime.date(year, 1, 1)
    d2 = datetime.date(year, 12, 31)
    dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]
    
    all_icetypes = np.full((len(dates),84,103),np.nan)
    
    file = Dataset(f'{datapath_in}iceage_nh_12.5km_{year}0101_{year}1231_v4.1.nc')
    lons = np.array(file['longitude'])
    lats = np.array(file['latitude'])
    icetypes = np.array(file['age_of_sea_ice'])
    icetype_grid_25 = regrid_fast(icetypes,lons,lats,ease_lons,ease_lats)
    
    # turn weekly data into daily for easy use
    for count,d in enumerate(np.arange(0,len(dates))):
        week = int(count / 7)
        if week == 52:
            week = 51
        all_icetypes[d] = icetype_grid_25[week]   
        
    all_icetypes[all_icetypes<1] = np.nan
    all_icetypes[all_icetypes>16] = np.nan
    fyi = np.where((all_icetypes>=1) & (all_icetypes<2))
    all_icetypes[fyi] = 2
    all_icetypes[all_icetypes>2] = 3
    
    dates = [date.strftime('%Y%m%d') for date in dates]
    
    filepath_out = f'{datapath_aux}icetype/{year}.nc'
    if os.path.exists(filepath_out) == True: 
        ! rm {filepath_out}

    ### create icetype NETCDF
    data_vars = {'icetype':(['t','x','y'],all_icetypes)}
    ds = xr.Dataset(data_vars = data_vars,
                    coords={'longitude':(['x','y'],ease_lons),
                            'latitude':(['x','y'],ease_lats),
                             'date':(['t'],dates)}     )
    ds.to_netcdf(filepath_out,'w')


  0%|                                                    | 0/13 [00:00<?, ?it/s]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/icetype/2010.nc: No such file or directory


  8%|███▍                                        | 1/13 [00:05<01:04,  5.34s/it]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/icetype/2011.nc: No such file or directory


 15%|██████▊                                     | 2/13 [00:10<00:59,  5.39s/it]

...

 85%|████████████████████████████████████▍      | 11/13 [00:59<00:10,  5.46s/it]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/icetype/2021.nc: No such file or directory


 92%|███████████████████████████████████████▋   | 12/13 [01:05<00:05,  5.45s/it]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/icetype/2022.nc: No such file or directory


100%|███████████████████████████████████████████| 13/13 [01:10<00:00,  5.44s/it]


### snow depth & density

In [9]:
filepath_in = '/Users/carmennab/Dropbox/alpha_retracker/data/snow/SMLG_MERRA2_01Jan2010-31Jul2021.nc'
snow_depths = np.array(Dataset(filepath_in)['Snow Depth'])
snow_densities = np.array(Dataset(filepath_in)['Snow Density'])
lons = np.load('/Users/carmennab/Dropbox/alpha_retracker/data/auxiliary/lon_25km_cent.npy')
lats = np.load('/Users/carmennab/Dropbox/alpha_retracker/data/auxiliary/lat_25km_cent.npy')

In [7]:
days = np.arange(0,len(snow_densities))
all_dates = [datetime.date(2010,1,1)+datetime.timedelta(days=int(days)) for days in days]

In [11]:
years = np.arange(2010,2023)

for year in tqdm.tqdm(years):
    
    d1 = datetime.date(year, 1, 1)
    d2 = datetime.date(year, 12, 31)
    dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]
    
    all_depths = np.full((len(dates),84,103),np.nan)
    all_densities = np.full((len(dates),84,103),np.nan)
    
    for date_count,date in enumerate(dates):
        
        for all_date_count,all_date in enumerate(all_dates):

            if all_date == date:
                all_depths[date_count] = regrid(snow_depths[all_date_count],lons,lats,ease_lons,ease_lats)
                all_densities[date_count] = regrid(snow_densities[all_date_count],lons,lats,ease_lons,ease_lats)

    dates = [date.strftime('%Y%m%d') for date in dates]
    
    ### create depth NETCDF
    filepath_out = f'{datapath_aux}snow_depth/{year}.nc'
    if os.path.exists(filepath_out) == True: 
        ! rm {filepath_out} 
    data_vars = {'snow_depth':(['t','x','y'],all_depths)}
    ds = xr.Dataset(data_vars = data_vars,
                    coords={'longitude':(['x','y'],ease_lons),
                            'latitude':(['x','y'],ease_lats),
                             'date':(['t'],dates)}     )
    ds.to_netcdf(filepath_out,'w')
    
     ### create density NETCDF
    filepath_out = f'{datapath_aux}snow_density/{year}.nc'
    if os.path.exists(filepath_out) == True: 
        ! rm {filepath_out}
    data_vars = {'snow_density':(['t','x','y'],all_densities)}
    ds = xr.Dataset(data_vars = data_vars,
                    coords={'longitude':(['x','y'],ease_lons),
                            'latitude':(['x','y'],ease_lats),
                             'date':(['t'],dates)}     )
    ds.to_netcdf(filepath_out,'w')

  0%|                                                    | 0/13 [00:00<?, ?it/s]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_depth/2010.nc: No such file or directory
rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_density/2010.nc: No such file or directory


  8%|███▍                                        | 1/13 [00:55<11:10, 55.91s/it]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_depth/2011.nc: No such file or directory
rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_density/2011.nc: No such file or directory


 15%|██████▊                                     | 2/13 [01:51<10:13, 55.79s/it]

...

 85%|████████████████████████████████████▍      | 11/13 [10:28<01:56, 58.05s/it]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_depth/2021.nc: No such file or directory
rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_density/2021.nc: No such file or directory


 92%|███████████████████████████████████████▋   | 12/13 [11:04<00:51, 51.36s/it]

rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_depth/2022.nc: No such file or directory
rm: /Users/carmennab/cpom_server/home/cjn/pb_new/auxiliary/snow_density/2022.nc: No such file or directory


100%|███████████████████████████████████████████| 13/13 [11:10<00:00, 51.56s/it]
